# 53 · Finance Analyst — Stripe + Contract PDF → Cash-Flow One-Pager

**Persona:** Finance analyst. **Tools exercised:** `StripeTool` (stubbed), `PDFTool` (real, against a PDF we generate in-notebook with reportlab), `DashboardRenderTool`.

Workflow:

1. Pull paid + open invoices from Stripe.
2. Generate a tiny contract PDF with reportlab inline, then extract text + metadata with the real `PDFTool`.
3. Stitch everything into a cash-flow one-pager dashboard.

Stripe is stubbed; PDF generation + extraction run for real if reportlab and pypdf are installed. If either dep is missing the notebook falls back to a pre-staged text blob and notes the skip.


## Setup

In [ ]:
from pathlib import Path

def _find_notebooks_dir() -> Path:
    cwd = Path.cwd().resolve()
    if cwd.name == 'notebooks':
        return cwd
    candidate = cwd / 'notebooks'
    return candidate if candidate.is_dir() else cwd
WORKSPACE = _find_notebooks_dir() / '_finance_workspace'
WORKSPACE.mkdir(parents=True, exist_ok=True)
print('workspace:', WORKSPACE)


## 1 · Pick a model

In [ ]:
# from shipit_agent.llms import build_llm_from_settings
# llm = build_llm_from_settings({'provider': 'bedrock',
#     'model': 'bedrock/anthropic.claude-sonnet-4-5-v2:0'}, provider='bedrock')
# llm = build_llm_from_settings({'provider': 'litellm',
#     'model': 'openai/gpt-4o-mini'}, provider='litellm')
# from shipit_agent.llms import LiteLLMProxyChatLLM
# llm = LiteLLMProxyChatLLM(model='gpt-4o-mini',
#     api_base='https://litellm.internal', api_key='sk-proxy')

from shipit_agent.llms import SimpleEchoLLM
llm = SimpleEchoLLM()
print('llm:', type(llm).__name__)


## 2 · Stubbed Stripe tool

In [ ]:
from shipit_agent.integrations import CredentialRecord, InMemoryCredentialStore
from shipit_agent.tools.stripe import StripeTool

store = InMemoryCredentialStore()
store.set(CredentialRecord(
    key='stripe', provider='stripe',
    secrets={'api_key': 'sk_test_demo'},
    metadata={'base_url': 'https://api.stripe.com'},
))
stripe = StripeTool(credential_store=store)

_INVOICES = [
    {'id': 'in_001', 'status': 'paid',  'amount_due': 199900, 'amount_paid': 199900,
     'currency': 'usd', 'customer': 'cus_NW', 'due_date': '2026-04-15'},
    {'id': 'in_002', 'status': 'paid',  'amount_due': 199900, 'amount_paid': 199900,
     'currency': 'usd', 'customer': 'cus_NW', 'due_date': '2026-04-15'},
    {'id': 'in_003', 'status': 'open',  'amount_due': 199900, 'amount_paid': 0,
     'currency': 'usd', 'customer': 'cus_ACME', 'due_date': '2026-04-30'},
    {'id': 'in_004', 'status': 'paid',  'amount_due':  99900, 'amount_paid':  99900,
     'currency': 'usd', 'customer': 'cus_BETA', 'due_date': '2026-04-18'},
    {'id': 'in_005', 'status': 'open',  'amount_due': 349900, 'amount_paid': 0,
     'currency': 'usd', 'customer': 'cus_GLOBEX', 'due_date': '2026-05-05'},
]

def _fake_stripe(*, record, method, path, query=None, body=None):
    if path == '/v1/invoices':
        return {'object': 'list', 'data': _INVOICES, 'has_more': False}
    return {}

stripe._request_json = _fake_stripe
print('stripe stub installed')


## 3 · Pull invoices through the tool

In [ ]:
from shipit_agent.tools.base import ToolContext

ctx = ToolContext(prompt='finance', state={'credential_store': store})

inv_out = stripe.run(ctx, action='list_invoices')
print(inv_out.text)

items = inv_out.metadata.get('items') or []
paid = [i for i in items if i['status'] == 'paid']
open_ = [i for i in items if i['status'] == 'open']
paid_usd = sum(i['amount_paid'] for i in paid) / 100
open_usd = sum(i['amount_due']  for i in open_) / 100
print(f'\npaid this month: ${paid_usd:,.0f}')
print(f'outstanding:     ${open_usd:,.0f}')


## 4 · Generate a contract PDF with reportlab

We write a tiny single-page PDF so the `PDFTool` has something real to extract against. If reportlab isn't installed we fall back to a text blob and note the skip.


In [ ]:
contract_pdf = WORKSPACE / 'northwind_contract.pdf'
reportlab_ok = False
try:
    from reportlab.lib.pagesizes import letter
    from reportlab.pdfgen import canvas
    reportlab_ok = True
except ImportError:
    print('reportlab not installed — falling back to staged text blob.')

if reportlab_ok:
    c = canvas.Canvas(str(contract_pdf), pagesize=letter)
    c.setTitle('Northwind Cloud — Master Subscription Agreement')
    c.setAuthor('Acme Finance')
    c.setFont('Helvetica-Bold', 16)
    c.drawString(72, 720, 'Master Subscription Agreement')
    c.setFont('Helvetica', 11)
    c.drawString(72, 696, 'Between: Acme Inc. ("Vendor") and Northwind Cloud Ltd. ("Customer")')
    c.drawString(72, 678, 'Effective date: 2026-04-01')
    c.drawString(72, 660, 'Term: 12 months, auto-renewing')
    c.drawString(72, 642, 'MRR: $1,999.00 USD — billed monthly in advance')
    c.drawString(72, 624, 'Invoice cadence: 1st of each month, Net 15')
    c.drawString(72, 606, 'Payment method: ACH (primary), card (fallback)')
    c.drawString(72, 588, 'Early-termination fee: 3 months MRR')
    c.drawString(72, 570, 'Total contract value (year 1): $23,988.00 USD')
    c.showPage()
    c.save()
    print(f'wrote {contract_pdf}  ({contract_pdf.stat().st_size} bytes)')
else:
    # Fallback fake — we'll skip the PDF extraction below.
    contract_pdf = None


## 5 · Extract contract text + metadata with the real PDFTool

In [ ]:
from shipit_agent.tools.pdf import PDFTool

pdf_tool = PDFTool()
pdf_text = ''
pdf_meta: dict = {}

if contract_pdf is not None:
    txt_out = pdf_tool.run(ctx, action='extract_text', source=str(contract_pdf))
    if txt_out.metadata.get('error'):
        print('PDFTool skipped:', txt_out.metadata['error'])
        print('  hint:', txt_out.metadata.get('install', ''))
    else:
        pdf_text = txt_out.text
        print('--- extracted text ---')
        print(pdf_text[:600])

    meta_out = pdf_tool.run(ctx, action='metadata', source=str(contract_pdf))
    if not meta_out.metadata.get('error'):
        pdf_meta = meta_out.metadata.get('metadata') or {}
        print()
        print('--- pdf metadata ---')
        for k, v in pdf_meta.items():
            print(f'{k}: {v}')
else:
    pdf_text = ('Master Subscription Agreement. Customer: Northwind Cloud. '
                'MRR $1,999 monthly Net 15. ACV $23,988.')
    print('(using staged text blob)')


## 6 · Cash-flow one-pager dashboard

In [ ]:
from shipit_agent.tools.dashboard_render import DashboardRenderTool

dash = DashboardRenderTool(workspace_root=WORKSPACE)

result = dash.run(
    ToolContext(prompt='finance',
                 state={'artifact_workspace_root': str(WORKSPACE)}),
    title='Cash-Flow One-Pager — April 2026',
    subtitle='Stripe invoices + Northwind MSA',
    lang='en',
    sections=[
        {'type': 'metrics', 'title': 'This month', 'columns': 3, 'items': [
            {'label': 'Collected', 'value': f'${paid_usd:,.0f}',
             'sub': f'{len(paid)} invoices'},
            {'label': 'Outstanding', 'value': f'${open_usd:,.0f}',
             'sub': f'{len(open_)} invoices', 'color': '#ba7517'},
            {'label': 'Net', 'value': f'${paid_usd - 0:,.0f}', 'sub': 'received'},
        ]},
        {'type': 'timeline', 'title': 'Invoice ledger', 'items': [
            {'period': i['due_date'],
             'head': f"{i['id']} — {i['customer']}",
             'desc': f"${i['amount_due']/100:,.0f} {i['currency'].upper()}  {i['status']}",
             'dot_color': '#1d9e75' if i['status'] == 'paid' else '#ba7517',
             'tags': [{'text': i['status'],
                        'color': 'green' if i['status'] == 'paid' else 'amber'}]}
            for i in items
        ]},
        {'type': 'cards', 'title': 'Key contract — Northwind MSA', 'columns': 1, 'cards': [
            {'title': pdf_meta.get('title') or 'Northwind MSA',
             'rows': [
                 {'strong': 'Extracted text:',
                  'text': (pdf_text or '(no extraction — see warning)')[:400],
                  'dot_color': '#185fa5'},
             ]}
        ]},
        {'type': 'verdict', 'title': 'Finance takeaway',
         'text': (f'**${paid_usd:,.0f}** collected, **${open_usd:,.0f}** '
                  f'outstanding. Chase Acme + Globex before month-end; '
                  f'Northwind MSA is ACV **${23988:,}** auto-renewing.')},
    ],
    export=True,
)
print(result.text)
print('artifact path:', result.metadata.get('path'))


## Next steps

* See `docs-app/content/source/tools/stripe.md` for the Stripe action surface (customers / subs / charges / catalog) and write-gating.
* See `docs-app/content/source/tools/pdf.md` for PDF extraction modes (`extract_text`, `extract_pages`, `metadata`, `page_count`).
* Wrap in `Autopilot(..., goal=Goal('Produce month-end cash-flow '
'one-pager'))` on a 1st-of-month schedule via the scheduler daemon.
